# WP3a: Optical Ice Classification (Colleague Track)

**Owner: colleague**

Two approaches:
1. **NDSI threshold** — fast baseline, no training data required.
2. **Random Forest** — requires manually digitised training polygons uploaded as a GEE asset.

Output band: `ice_s2` (1 = ice, 0 = open water/other) per image.

> **Hand-off to SAR track:** Export training polygons covering the 4 ice classes
> (open water, drift ice, landfast ice, glacier ice) as a GEE FeatureCollection asset.
> Julian needs this asset path to train the SVM in `03b_classification_s1.ipynb`.

In [ ]:
import ee
import geemap
import sys
sys.path.insert(0, '..')
from src.utils import load_aoi, get_gee_project
from src.preprocessing_s2 import preprocess_s2
from src.classification_s2 import classify_ndsi, train_random_forest, classify_rf, RF_BANDS

ee.Initialize(project=get_gee_project())
aoi = load_aoi()

## 3a.1 Load preprocessed S2 collection

In [ ]:
START, END = '2019-01-01', '2024-12-31'

s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START, END)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    .map(preprocess_s2)
)
print('S2 scenes (cloud < 30%):', s2.size().getInfo())

## 3a.2 NDSI threshold classification

Baseline: pixels with NDSI ≥ 0.4 are classified as ice.

In [ ]:
s2_classified = s2.map(classify_ndsi)

sample = s2_classified.first().clip(aoi)
Map = geemap.Map()
Map.centerObject(aoi, zoom=9)
Map.addLayer(sample, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'S2 True Colour')
Map.addLayer(
    sample.select('ice_s2'),
    {'min': 0, 'max': 1, 'palette': ['#1a6faf', 'white']},
    'Ice (NDSI threshold)'
)
Map

## 3a.3 Random Forest classifier

Requires training polygons digitised in GEE and uploaded as a FeatureCollection asset.
Each polygon needs a `class` property with integer values:

| Value | Class |
|---|---|
| 0 | Open water |
| 1 | Drift ice |
| 2 | Landfast ice |
| 3 | Glacier ice |

Use S2 true-colour and NDSI layers above to guide digitising.
Export the asset path and share it with Julian for SVM training.

In [ ]:
# TODO: replace with your GEE asset path after uploading training polygons
# TRAINING_ASSET = 'projects/<your-project>/assets/sermilik_training_polygons'
#
# training_fc = ee.FeatureCollection(TRAINING_ASSET)
# classifier = train_random_forest(training_fc, bands=RF_BANDS)
# s2_rf = s2.map(lambda img: classify_rf(img, classifier))
print('RF classification: upload training polygons and uncomment the block above.')